# GPU timing

> **Install for this page:** `pip install scope-profiler`; CuPy or PyTorch for real
> CUDA timing.

GPU work is asynchronous: a region around a CuPy kernel launch measures the
host side, i.e. how long it took to *enqueue* the work. scope-profiler can
additionally record CUDA events around each region and store the device time as
the `gpu_total` and `gpu_avg` statistics.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Work in a scratch directory so nothing lands in the working directory,
# and make the `scope-profiler` command of this environment available.
WORKDIR = Path(tempfile.mkdtemp(prefix="scope-profiler-guide-"))
os.chdir(WORKDIR)
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
os.environ["PYTHONPATH"] = os.pathsep.join(
    filter(None, [str(WORKDIR), os.environ.get("PYTHONPATH")])
)
sys.path.insert(0, str(WORKDIR))

## Turning it on

Set `use_gpu_timing` and pick the backend (`"auto"`, `"cupy"` or `"torch"`):

```python
import cupy as cp
import scope_profiler as sp

with sp.session(output="gpu.h5", use_gpu_timing=True, gpu_timing_backend="cupy"):
    for _ in range(100):
        with sp.region("step"):
            y = cp.sin(x) @ x
```

The CPU timestamps are still recorded, so the timeline stays the enqueue-side
view; `gpu_sync_on_exit=True` makes each region wait for the device, which
gives host times that include the device work at the price of removing the
overlap between CPU and GPU. For device times of many short kernels, wrap a loop
of steps in one region rather than every kernel. For a timeline of the
kernels themselves, add NVTX ranges (`use_nvtx=True`, `pip install
scope-profiler[nvtx]`) and record with Nsight Systems.

## A custom backend

The backend can also be any object with `record_event()` and
`elapsed_time_ns(start, end)`. This page has no GPU, so a fake backend that
reports host time stands in for CUDA events and shows where the device time
ends up:

In [ ]:
import time

import scope_profiler as sp


class HostClockBackend:
    """Stand-in for CUDA events: an event is a host timestamp."""

    name = "host"

    def record_event(self):
        return time.perf_counter_ns()

    def elapsed_time_ns(self, start_event, end_event):
        return end_event - start_event


with sp.session(
    output="gpu.h5",
    use_gpu_timing=True,
    gpu_timing_backend=HostClockBackend(),
    verbose=False,
):
    for _ in range(3):
        with sp.region("step"):
            time.sleep(0.002)

In [ ]:
%%bash
scope-profiler inspect gpu.h5 --regions-only --columns region calls total gpu_total gpu_avg